In [10]:
"""
STEP 1: Exploratory Data Analysis (EDA)

The EDA looks at the data before any modelling begins. It answers five
questions:

  1. How much data is there, and is any of it missing?
  2. How imbalanced is it (healthy vs bankrupt companies)?
  3. Do bankrupt and healthy companies actually look different?
  4. Has the bankruptcy rate changed over the years?
  5. How do the financial figures relate to one another?

No models are trained here. This is purely getting to know the data.
"""

import pandas as pd
import matplotlib
matplotlib.use("Agg")   # lets charts save to file without opening a window
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("bankruptcy_data.csv")
feature_columns = [c for c in df.columns if c.startswith("X")]

#  The 18 financial figures in the dataset
FEATURE_LABELS = {
    "X1": "Current Assets", "X2": "Cost of Goods Sold",
    "X3": "Depreciation and Amortization", "X4": "EBITDA",
    "X5": "Inventory", "X6": "Net Income", "X7": "Total Receivables",
    "X8": "Market Value", "X9": "Net Sales", "X10": "Total Assets",
    "X11": "Total Long-Term Debt", "X12": "EBIT", "X13": "Gross Profit",
    "X14": "Total Current Liabilities", "X15": "Retained Earnings",
    "X16": "Total Revenue", "X17": "Total Liabilities",
    "X18": "Total Operating Expenses",
}


# QUESTION 1: How much data is there, and is any missing?

print("=" * 65)
print("1. BASIC SHAPE OF THE DATA")
print("=" * 65)
print(f"Rows (company-year records): {df.shape[0]:,}")
print(f"Unique companies:            {df['company_name'].nunique():,}")
print(f"Years covered:               {df['year'].min()} to {df['year'].max()}")
print(f"Financial figures per row:   {len(feature_columns)}")
print(f"Missing values anywhere:     {df.isnull().sum().sum()}")


# QUESTION 2: How imbalanced is the data?

print("\n" + "=" * 65)
print("2. CLASS BALANCE (the imbalance problem)")
print("=" * 65)
counts = df["status_label"].value_counts()
percentages = df["status_label"].value_counts(normalize=True) * 100
for label in counts.index:
    print(f"{label:8s}: {counts[label]:6,} records ({percentages[label]:5.2f}%)")

plt.figure(figsize=(5, 4))
counts.plot(kind="bar", color=["#3B4A9E", "#C0392B"])
plt.title("Number of Companies: Alive vs Failed", fontweight="bold")
plt.ylabel("Company-year records")
plt.xticks(rotation=0)
plt.tight_layout()
plt.savefig("chart_class_balance.png", dpi=150)
plt.close()
print("\nSaved: chart_class_balance.png")


# QUESTION 3: Do bankrupt and healthy companies actually look different?

# This is the most important question in the whole EDA. If the two groups
# looked identical on these figures, no model could ever separate them.
print("\n" + "=" * 65)
print("3. HEALTHY vs BANKRUPT COMPANIES COMPARED")
print("=" * 65)
print(f"{'Financial figure':32s} {'Alive':>12s} {'Failed':>12s}")
print("-" * 65)
comparison_features = ["X6", "X15", "X11", "X17", "X10", "X1"]
for feat in comparison_features:
    alive_median = df[df["status_label"] == "alive"][feat].median()
    failed_median = df[df["status_label"] == "failed"][feat].median()
    name = f"{feat} - {FEATURE_LABELS[feat]}"
    print(f"{name:32s} {alive_median:12.2f} {failed_median:12.2f}")

chart_features = ["X6", "X15", "X11", "X17"]
fig, axes = plt.subplots(1, 4, figsize=(14, 3.6))
for ax, feat in zip(axes, chart_features):
    alive_median = df[df["status_label"] == "alive"][feat].median()
    failed_median = df[df["status_label"] == "failed"][feat].median()
    ax.bar(["Alive", "Failed"], [alive_median, failed_median], color=["#3B4A9E", "#C0392B"])
    ax.set_title(f"{feat}\n{FEATURE_LABELS[feat]}", fontsize=10)
    ax.axhline(0, color="black", linewidth=0.8)
    if feat == chart_features[0]:
        ax.set_ylabel("Median value")
plt.suptitle("Median Financial Figures: Healthy vs Bankrupt Companies", fontweight="bold")
plt.tight_layout()
plt.savefig("chart_group_comparison.png", dpi=150)
plt.close()
print("\nSaved: chart_group_comparison.png")


# QUESTION 4: Has the bankruptcy rate changed over the years?

# This matters because the data is split by year for training and testing.
# If the failure rate is not stable, the training years and the test years
# are not equally difficult.
print("\n" + "=" * 65)
print("4. BANKRUPTCY RATE OVER TIME, AND ACROSS THE THREE SPLITS")
print("=" * 65)
for name, first_year, last_year in [("Training", 1999, 2011),
                                     ("Validation", 2012, 2014),
                                     ("Test", 2015, 2018)]:
    subset = df[(df["year"] >= first_year) & (df["year"] <= last_year)]
    rate = (subset["status_label"] == "failed").mean() * 100
    print(f"{name:11s} ({first_year}-{last_year}): {len(subset):6,} records, {rate:5.2f}% failed")

print("\nThe failure rate falls steadily across the three periods. The test set is")
print("therefore harder and more imbalanced than the data the models learn from,")
print("which means the reported test results are a conservative estimate.")

plt.figure(figsize=(9, 4))
yearly_rate = df.groupby("year")["status_label"].apply(lambda s: (s == "failed").mean() * 100)
plt.plot(yearly_rate.index, yearly_rate.values, marker="o", color="#C0392B", linewidth=2)
plt.axvspan(1999, 2011, alpha=0.12, color="#1E2761", label="Training years")
plt.axvspan(2012, 2014, alpha=0.12, color="#3B4A9E", label="Validation years")
plt.axvspan(2015, 2018, alpha=0.12, color="#2E7D32", label="Test years")
plt.title("Bankruptcy Rate by Year", fontweight="bold")
plt.ylabel("% of companies that failed")
plt.xlabel("Year")
plt.legend(fontsize=8)
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("chart_bankruptcy_rate_over_time.png", dpi=150)
plt.close()
print("\nSaved: chart_bankruptcy_rate_over_time.png")

plt.figure(figsize=(8, 4))
df["year"].value_counts().sort_index().plot(kind="bar", color="#3B4A9E")
plt.title("Number of Company Records per Year", fontweight="bold")
plt.ylabel("Records")
plt.xlabel("Year")
plt.tight_layout()
plt.savefig("chart_records_per_year.png", dpi=150)
plt.close()
print("Saved: chart_records_per_year.png")


# QUESTION 5: How do the financial figures relate to one another?
# A heatmap shows which figures move together. Values near 1 (dark red) mean
# two figures rise and fall together; near 0 (pale) means they are unrelated.
# Strongly related figures carry overlapping information, which is part of
# why the feature selection step later on is worthwhile.
print("\n" + "=" * 65)
print("5. HOW THE FINANCIAL FIGURES RELATE TO ONE ANOTHER")
print("=" * 65)
correlation_matrix = df[feature_columns].corr()

plt.figure(figsize=(11, 9))
sns.heatmap(correlation_matrix, annot=True, fmt=".2f", cmap="coolwarm",
            center=0, square=True, cbar_kws={"label": "Correlation"},
            annot_kws={"size": 7})
plt.title("Correlation Heatmap of the 18 Financial Features", fontweight="bold")
plt.tight_layout()
plt.savefig("chart_correlation_heatmap.png", dpi=150)
plt.close()
print("Saved: chart_correlation_heatmap.png")

# Report the most strongly related pairs
pairs = correlation_matrix.abs().unstack().sort_values(ascending=False)
pairs = pairs[pairs < 0.999]          # drop each feature paired with itself
seen, top_pairs = set(), []
for (a, b), value in pairs.items():
    if (b, a) not in seen:
        seen.add((a, b))
        top_pairs.append((a, b, value))
    if len(top_pairs) == 3:
        break
print("\nMost strongly related pairs of figures:")
for a, b, value in top_pairs:
    print(f"  {a} & {b}  ({FEATURE_LABELS[a]} / {FEATURE_LABELS[b]}): {value:.2f}")

print("\n" + "=" * 65)
print("EDA complete. Five charts saved. Move on to Step 2.")
print("=" * 65)


1. BASIC SHAPE OF THE DATA
Rows (company-year records): 78,682
Unique companies:            8,971
Years covered:               1999 to 2018
Financial figures per row:   18
Missing values anywhere:     0

2. CLASS BALANCE (the imbalance problem)
alive   : 73,462 records (93.37%)
failed  :  5,220 records ( 6.63%)

Saved: chart_class_balance.png

3. HEALTHY vs BANKRUPT COMPANIES COMPARED
Financial figure                        Alive       Failed
-----------------------------------------------------------------
X6 - Net Income                          2.07        -3.33
X15 - Retained Earnings                 -0.17       -25.51
X11 - Total Long-Term Debt               7.09        18.44
X17 - Total Liabilities                 80.74        97.03
X10 - Total Assets                     215.01       195.14
X1 - Current Assets                    102.92        75.87

Saved: chart_group_comparison.png

4. BANKRUPTCY RATE OVER TIME, AND ACROSS THE THREE SPLITS
Training    (1999-2011): 55,927 records